# Phase 5 — DSPy Optimization + QLoRA Fine-Tuning (Tier 3, N=10)
Tier 3 baseline (no optimization) was 90.0% (9/10) — closer to Tier 1's ceiling than Tier 2's headroom, but this is the tier your paper is currently missing a full DSPy-vs-QLoRA comparison for. Closing this gap matters more for reviewers than adding seeds elsewhere.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show a Tesla T4 GPU table with 0MiB used.**

## 1. Clone repo (safe to re-run any time)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pwd

In [ ]:
!grep -c "def sample_tier3_training" envs/training_data.py
!grep -c "from dspy_optimize_tier2 import ChainProgram" dspy_optimize_tier3.py
!grep -c "def build_chat_example" qlora_finetune_tier3.py

Each of the 3 grep counts above should print `1`. If any print `0`, the file wasn't pushed correctly.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai optuna

In [ ]:
from huggingface_hub import login
login()

## 2. Sanity-check the Tier 3 training data generator (no GPU needed)

In [ ]:
!python envs/training_data.py

Check the TIER 3 section near the bottom: pool size ~66, leakage check = 0.

## 3. Load model + run DSPy optimization

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier3_training
from tasks.tier3 import TIER3_HELDOUT
import dspy_optimize_tier3

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model + DSPy LM wrapper ready.")

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

train_10 = sample_tier3_training(10)
print(f"Training on {len(train_10)} Tier 3 examples.")

optimized_program = dspy_optimize_tier3.optimize(lm, train_10)
print("DSPy optimization complete.")

In [ ]:
dspy_results = dspy_optimize_tier3.evaluate_program(optimized_program, TIER3_HELDOUT)
for r in dspy_results:
    print(f"[{'PASS' if r['grade']['success'] else 'FAIL'}] {r['id']} — {r['grade'].get('failure_type')}")

dspy_success_rate = sum(r["grade"]["success"] for r in dspy_results) / len(dspy_results)
print(f"\nDSPy-optimized Tier 3 (N=10) held-out success rate: {dspy_success_rate:.1%}")

with open("results/tier3_dspy_n10_results.json", "w") as f:
    json.dump(dspy_results, f, indent=2)

## 4. Download DSPy results NOW, before continuing

In [ ]:
from google.colab import files
files.download("results/tier3_dspy_n10_results.json")

## 5. Restart the runtime before QLoRA
This is not optional — running QLoRA in the same session as DSPy caused OOM crashes in earlier phases. Runtime -> Restart session (or Disconnect and delete runtime), then re-run: env var cell, GPU check, Section 1 (clone/verify/install/login). Skip straight to Section 6 after that.

## 6. QLoRA fine-tuning (N=10)

In [ ]:
!python qlora_finetune_tier3.py --n 10

## 7. Evaluate the fine-tuned adapter
Uses the same multi-turn agent harness as the Phase 2 baseline, so the adapter is tested the way it would actually be used.

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier3 import TIER3_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
ft_model = PeftModel.from_pretrained(base_model, "adapters/tier3_n10")
ft_tok = AutoTokenizer.from_pretrained("adapters/tier3_n10")
print("Fine-tuned model loaded.")

In [ ]:
qlora_results = []
for task in TIER3_HELDOUT:
    tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=3, tool_calls=tool_calls, final_text=final_text)
    qlora_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

qlora_success_rate = sum(r["grade"]["success"] for r in qlora_results) / len(qlora_results)
print(f"\nQLoRA fine-tuned Tier 3 (N=10) held-out success rate: {qlora_success_rate:.1%}")

with open("results/tier3_qlora_n10_results.json", "w") as f:
    json.dump(qlora_results, f, indent=2)

## 8. Download everything and push

In [ ]:
from google.colab import files
files.download("results/tier3_qlora_n10_results.json")
files.download("adapters/tier3_n10/training_examples.json")

Move both into `results/` on your laptop (rename `training_examples.json` to `tier3_training_examples_n10.json`), then:
```bash
git add results/tier3_dspy_n10_results.json results/tier3_qlora_n10_results.json results/tier3_training_examples_n10.json
git commit -m "Phase 5: DSPy + QLoRA results for Tier 3, N=10"
git push
```